# Long Short-Term Memory — Implementation
# 장단기 기억 — 구현

**Paper**: Hochreiter, S. & Schmidhuber, J. (1997). "Long Short-Term Memory." *Neural Computation*, 9(8), 1735–1780.

이 노트북은 LSTM의 핵심 개념을 처음부터 구현하고, 기울기 소실 문제와 LSTM의 해결책을 시각화합니다.
This notebook implements LSTM's core concepts from scratch and visualizes the vanishing gradient problem and LSTM's solution.

**구성 / Structure:**
1. **Part 1**: Vanishing Gradient 시각화 — 왜 RNN이 실패하는가 / Why RNNs Fail
2. **Part 2**: LSTM Cell from Scratch — 순전파 구현 / Forward Pass Implementation
3. **Part 3**: LSTM vs Vanilla RNN — 장기 의존성 학습 비교 / Long-range Dependency Comparison
4. **Part 4**: Gate Dynamics 시각화 — 게이트가 어떻게 작동하는가 / How Gates Work
5. **Part 5**: Adding Problem 재현 (논문 실험 4) / Reproducing the Adding Problem (Experiment 4)
6. **Part 6**: Sequence Classification — PyTorch LSTM / Practical Sequence Classification
7. **Summary**: 핵심 개념 정리 및 다음 논문 연결 / Key Concepts and Next Papers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
np.random.seed(42)
torch.manual_seed(42)

## Part 1: Vanishing Gradient 시각화 / Visualizing the Vanishing Gradient

논문 Section 3.1 (Eq. 2): 오류가 시간을 거슬러 전파될 때, 각 스텝에서 $f'(net) \cdot w$가 곱해집니다.
Section 3.1 (Eq. 2): Error is multiplied by $f'(net) \cdot w$ at each step when propagated back through time.

$$\frac{\partial \vartheta_v(t-q)}{\partial \vartheta_u(t)} \propto \prod_{m=1}^{q} f'_{l_m}(net_{l_m}(t-m)) \cdot w_{l_m l_{m-1}}$$

시그모이드의 $f'_{\max} = 0.25$이므로, $|w| < 4$이면 **기울기가 기하급수적으로 소실**합니다.
Since $f'_{\max} = 0.25$ for sigmoid, if $|w| < 4$ then **gradients vanish exponentially**.

In [ ]:
def sigmoid(x):
    """Logistic sigmoid function."""
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    """Derivative of sigmoid: f'(x) = f(x)(1 - f(x)). Max = 0.25."""
    s = sigmoid(x)
    return s * (1 - s)

# Visualize gradient flow over time steps
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Sigmoid and its derivative
x = np.linspace(-6, 6, 200)
axes[0].plot(x, sigmoid(x), "b-", linewidth=2, label=r"$\sigma(x)$")
axes[0].plot(x, sigmoid_derivative(x), "r--", linewidth=2, label=r"$\sigma'(x)$")
axes[0].axhline(0.25, color="gray", linestyle=":", alpha=0.5)
axes[0].annotate(r"$f'_{max} = 0.25$", xy=(2, 0.25), fontsize=11, color="gray")
axes[0].set_title("Sigmoid and its Derivative\n시그모이드와 미분", fontsize=13)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# Plot 2: Gradient magnitude vs time steps (Eq. 2 product)
time_steps = np.arange(1, 101)
weights = [0.5, 1.0, 2.0, 3.0, 4.0]
for w in weights:
    # |f'(net) * w| at maximum: 0.25 * |w|
    scale_factor = 0.25 * abs(w)
    gradient_magnitude = scale_factor ** time_steps
    axes[1].semilogy(time_steps, gradient_magnitude, linewidth=2,
                     label=f"|w|={w}, scale={scale_factor:.2f}")

axes[1].axhline(1.0, color="black", linestyle="--", alpha=0.3)
axes[1].set_xlabel("Time steps back (q)")
axes[1].set_ylabel("Gradient magnitude (log scale)")
axes[1].set_title("Gradient Flow: $(0.25 \\times |w|)^q$\n기울기 흐름: 시간에 따른 기하급수적 변화")
axes[1].legend(fontsize=9)
axes[1].set_ylim(1e-30, 1e30)
axes[1].grid(True, alpha=0.3)

# Plot 3: Compare vanilla RNN gradient vs LSTM (CEC) gradient
q_vals = np.arange(1, 51)
rnn_gradient = (0.25 * 2.0) ** q_vals  # Vanilla RNN with w=2
lstm_gradient = np.ones_like(q_vals, dtype=float)  # CEC: constant!

axes[2].semilogy(q_vals, rnn_gradient, "r-", linewidth=2, label="Vanilla RNN (w=2, scale=0.5)")
axes[2].semilogy(q_vals, lstm_gradient, "g-", linewidth=3, label="LSTM CEC (constant = 1.0)")
axes[2].fill_between(q_vals, rnn_gradient, lstm_gradient, alpha=0.1, color="red")
axes[2].set_xlabel("Time steps back (q)")
axes[2].set_ylabel("Gradient magnitude")
axes[2].set_title("RNN vs LSTM: Gradient Flow\nRNN vs LSTM: 기울기 흐름 비교")
axes[2].legend(fontsize=11)
axes[2].set_ylim(1e-15, 10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight (핵심 통찰):")
print("  RNN: gradient → 0 exponentially (vanishing)")
print("  LSTM CEC: gradient stays constant at 1.0 (no vanishing!)")

## Part 2: LSTM Cell from Scratch
## 파트 2: LSTM 셀 직접 구현

논문 Section 4의 수식을 NumPy로 직접 구현합니다 (Figure 1의 아키텍처).
Implementing the equations from Section 4 in NumPy (Figure 1 architecture).

**핵심 수식 / Key equations:**
- $s_{c_j}(t) = s_{c_j}(t-1) + y^{in_j}(t) \cdot g(net_{c_j}(t))$ — Cell state update (CEC!)
- $y^{c_j}(t) = y^{out_j}(t) \cdot h(s_{c_j}(t))$ — Cell output

In [ ]:
class LSTMCellFromScratch:
    """Original 1997 LSTM cell (no forget gate).

    Implements:
      input gate:  y_in(t)  = sigma(W_in @ concat(x, h) + b_in)
      cell input:  g_val    = g(W_c @ concat(x, h) + b_c)
      cell state:  s(t)     = s(t-1) + y_in(t) * g_val     # CEC!
      output gate: y_out(t) = sigma(W_out @ concat(x, h) + b_out)
      cell output: y_c(t)   = y_out(t) * h(s(t))
    """

    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        concat_size = input_size + hidden_size

        # Xavier initialization
        scale = np.sqrt(2.0 / concat_size)
        self.W_in = np.random.randn(hidden_size, concat_size) * scale
        self.W_out = np.random.randn(hidden_size, concat_size) * scale
        self.W_c = np.random.randn(hidden_size, concat_size) * scale

        # Bias: input gate slightly negative, output gate more negative
        # (following paper's abuse prevention strategy)
        self.b_in = np.full(hidden_size, -1.0)
        self.b_out = np.full(hidden_size, -2.0)
        self.b_c = np.zeros(hidden_size)

    def _g(self, x):
        """Input squashing: scaled sigmoid, range [-2, 2]."""
        return 4.0 * sigmoid(x) - 2.0

    def _h(self, x):
        """Output squashing: scaled sigmoid, range [-1, 1]."""
        return 2.0 * sigmoid(x) - 1.0

    def forward(self, x_sequence):
        """Run LSTM on a full sequence.

        Args:
            x_sequence: Input sequence, shape (seq_len, input_size).

        Returns:
            outputs: Cell outputs at each step, shape (seq_len, hidden_size).
            states: Cell states at each step, shape (seq_len, hidden_size).
            gates: Dict of gate activations for visualization.
        """
        seq_len = len(x_sequence)
        h = np.zeros(self.hidden_size)
        s = np.zeros(self.hidden_size)  # Cell state, init to 0

        outputs = np.zeros((seq_len, self.hidden_size))
        states = np.zeros((seq_len, self.hidden_size))
        input_gates = np.zeros((seq_len, self.hidden_size))
        output_gates = np.zeros((seq_len, self.hidden_size))

        for t in range(seq_len):
            concat = np.concatenate([x_sequence[t], h])

            # Gate activations
            y_in = sigmoid(self.W_in @ concat + self.b_in)    # Input gate
            y_out = sigmoid(self.W_out @ concat + self.b_out)  # Output gate

            # Cell input (squashed)
            g_val = self._g(self.W_c @ concat + self.b_c)

            # ★ CEC: cell state = previous state + gated input ★
            s = s + y_in * g_val

            # Cell output = output gate * h(cell state)
            h = y_out * self._h(s)

            outputs[t] = h
            states[t] = s
            input_gates[t] = y_in
            output_gates[t] = y_out

        return outputs, states, {
            "input_gates": input_gates,
            "output_gates": output_gates,
        }


# Demo: run on a simple sequence
lstm_cell = LSTMCellFromScratch(input_size=3, hidden_size=4)
x_demo = np.random.randn(20, 3)  # 20 time steps, 3 features
outputs, states, gates = lstm_cell.forward(x_demo)

print(f"Input shape:  {x_demo.shape}")
print(f"Output shape: {outputs.shape}")
print(f"States shape: {states.shape}")
print(f"\nCell state at t=0:  {states[0, :2]}")
print(f"Cell state at t=19: {states[19, :2]}")
print("→ Cell state accumulates over time (CEC property!)")

## Part 3: LSTM vs Vanilla RNN — 장기 의존성 학습 / Long-range Dependency Learning

논문 Table 2의 핵심 결과를 재현: Vanilla RNN이 긴 time lag에서 실패하고 LSTM이 성공하는 것을 보여줍니다.
Reproducing the key result from Table 2: Vanilla RNN fails at long time lags while LSTM succeeds.

**작업**: 시퀀스의 첫 원소를 기억하여 마지막에 재생. "Recall first element" 작업.
**Task**: Remember the first element of a sequence and reproduce it at the end. "Recall first element" task.

In [ ]:
def generate_recall_data(n_samples, seq_len, n_classes=8):
    """Generate 'recall first element' task data.

    Sequence: [class_signal, noise, noise, ..., noise]
    Target: class of the first element (after seq_len steps).
    """
    X = torch.zeros(n_samples, seq_len, n_classes + 1)
    y = torch.zeros(n_samples, dtype=torch.long)

    for i in range(n_samples):
        label = np.random.randint(0, n_classes)
        y[i] = label
        # First element: one-hot class signal
        X[i, 0, label] = 1.0
        # Remaining elements: random noise in the extra channel
        X[i, 1:, -1] = torch.randn(seq_len - 1) * 0.1

    return X, y


class SimpleRNN(nn.Module):
    """Vanilla RNN for classification."""

    def __init__(self, input_size, hidden_size, n_classes):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        _, h_n = self.rnn(x)
        return self.fc(h_n.squeeze(0))


class SimpleLSTM(nn.Module):
    """LSTM for classification."""

    def __init__(self, input_size, hidden_size, n_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n.squeeze(0))


def train_and_evaluate(model, X_train, y_train, X_test, y_test,
                       epochs=200, lr=0.01):
    """Train a model and return accuracy history."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    acc_history = []

    dataset = TensorDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=64, shuffle=True)

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            pred = model(X_test).argmax(dim=1)
            acc = (pred == y_test).float().mean().item()
            acc_history.append(acc)

    return acc_history


# Compare RNN vs LSTM at different sequence lengths (time lags)
seq_lengths = [5, 20, 50, 100]
n_classes = 8
hidden_size = 32
n_train, n_test = 1000, 200

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, seq_len in zip(axes.flat, seq_lengths):
    torch.manual_seed(42)
    X_tr, y_tr = generate_recall_data(n_train, seq_len, n_classes)
    X_te, y_te = generate_recall_data(n_test, seq_len, n_classes)
    input_size = n_classes + 1

    # Vanilla RNN
    rnn_model = SimpleRNN(input_size, hidden_size, n_classes)
    rnn_acc = train_and_evaluate(rnn_model, X_tr, y_tr, X_te, y_te, epochs=150)

    # LSTM
    lstm_model = SimpleLSTM(input_size, hidden_size, n_classes)
    lstm_acc = train_and_evaluate(lstm_model, X_tr, y_tr, X_te, y_te, epochs=150)

    ax.plot(rnn_acc, "r-", alpha=0.7, label=f"Vanilla RNN ({rnn_acc[-1]:.0%})")
    ax.plot(lstm_acc, "g-", alpha=0.7, label=f"LSTM ({lstm_acc[-1]:.0%})")
    ax.axhline(1.0 / n_classes, color="gray", linestyle=":", alpha=0.5, label="Random chance")
    ax.set_title(f"Time lag = {seq_len}", fontsize=13)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Test Accuracy")
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

plt.suptitle("RNN vs LSTM: Recall First Element Task (Table 2 style)\n"
             "RNN vs LSTM: 첫 원소 기억 작업", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 4: Gate Dynamics 시각화 / Visualizing Gate Dynamics

입력 게이트와 출력 게이트가 어떻게 정보 흐름을 제어하는지 시각화합니다.
Visualizing how input and output gates control information flow.

논문의 핵심 통찰: 게이트 ≈ 0이면 정보 차단, 게이트 ≈ 1이면 정보 통과.
Paper's key insight: gate ≈ 0 blocks information, gate ≈ 1 passes information.

In [ ]:
# Visualize gate dynamics on a trained LSTM
# Use the recall task with seq_len=50
torch.manual_seed(42)
seq_len = 50
X_vis, y_vis = generate_recall_data(1, seq_len, n_classes=8)

# Train a small LSTM
lstm_vis = SimpleLSTM(9, 16, 8)
X_tr_vis, y_tr_vis = generate_recall_data(500, seq_len, 8)
_ = train_and_evaluate(lstm_vis, X_tr_vis, y_tr_vis, X_vis, y_vis, epochs=100)

# Extract gate activations using hooks
gate_data = {}

def hook_fn(module, input, output):
    """Hook to capture LSTM internal states."""
    # output = (output_seq, (h_n, c_n))
    gate_data["output_seq"] = output[0].detach().numpy()
    gate_data["h_n"] = output[1][0].detach().numpy()
    gate_data["c_n"] = output[1][1].detach().numpy()

hook = lstm_vis.lstm.register_forward_hook(hook_fn)
with torch.no_grad():
    pred = lstm_vis(X_vis)
hook.remove()

# Get full sequence of hidden states
full_output = gate_data["output_seq"][0]  # (seq_len, hidden_size)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Plot 1: Input signal
ax = axes[0]
input_signal = X_vis[0].numpy()
im = ax.imshow(input_signal.T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_ylabel("Input channels")
ax.set_title(f"Input Sequence (class={y_vis[0].item()}, signal at t=0 only)\n"
             f"입력 시퀀스 (클래스={y_vis[0].item()}, 신호는 t=0에만)", fontsize=12)
plt.colorbar(im, ax=ax, shrink=0.6)

# Plot 2: Hidden state (output) over time
ax = axes[1]
im = ax.imshow(full_output.T, aspect="auto", cmap="viridis")
ax.set_ylabel("Hidden units")
ax.set_title("LSTM Hidden State Over Time / 시간에 따른 LSTM 은닉 상태", fontsize=12)
plt.colorbar(im, ax=ax, shrink=0.6)

# Plot 3: Selected hidden unit activations
ax = axes[2]
for i in range(min(4, full_output.shape[1])):
    ax.plot(full_output[:, i], linewidth=1.5, alpha=0.7, label=f"Unit {i}")
ax.axvline(0, color="red", linestyle="--", alpha=0.5, label="Signal at t=0")
ax.set_xlabel("Time step")
ax.set_ylabel("Activation")
ax.set_title("Hidden Unit Traces / 은닉 유닛 활성화 추이", fontsize=12)
ax.legend(fontsize=9, ncol=3)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Prediction: class {pred.argmax().item()}, True: class {y_vis[0].item()}")
print("Observation: LSTM maintains distinct hidden state patterns")
print("from t=0 signal through the entire noisy sequence.")

## Part 5: Adding Problem 재현 / Reproducing the Adding Problem (Experiment 4)

논문 Section 5.4: 시퀀스의 마커된 두 위치의 값의 합을 출력. 분산된 연속값 표상 + 장기 저장을 동시에 요구.
Section 5.4: Output the sum of values at two marked positions. Requires distributed continuous-valued representation + long-term storage.

**구조**: 각 입력 = (value, marker). marker ∈ {-1, 0, 1}. 두 위치만 marker=1, 나머지는 marker=0 (또는 -1).
**Structure**: Each input = (value, marker). marker ∈ {-1, 0, 1}. Only two positions have marker=1.

In [ ]:
def generate_adding_problem(n_samples, seq_len):
    """Generate the Adding Problem data (paper Section 5.4).

    Each element: (random_value in [-1,1], marker in {0, 1, -1}).
    Two positions are marked (marker=1). Target = sum of their values,
    scaled to [0, 1]: target = 0.5 + (X1 + X2) / 4.0.
    """
    X = torch.zeros(n_samples, seq_len, 2)
    y = torch.zeros(n_samples, 1)

    for i in range(n_samples):
        # Random values in [-1, 1]
        values = torch.FloatTensor(seq_len).uniform_(-1, 1)
        X[i, :, 0] = values

        # Markers: all -1 except first and last pairs
        X[i, :, 1] = -1.0

        # Mark two positions: one in first half, one in second half
        pos1 = np.random.randint(0, seq_len // 2)
        pos2 = np.random.randint(seq_len // 2, seq_len)
        X[i, pos1, 1] = 1.0
        X[i, pos2, 1] = 1.0

        # Target: scaled sum
        y[i] = 0.5 + (values[pos1] + values[pos2]) / 4.0

    return X, y


class AddingLSTM(nn.Module):
    """LSTM for the Adding Problem (regression)."""

    def __init__(self, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(2, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return torch.sigmoid(self.fc(h_n.squeeze(0)))


class AddingRNN(nn.Module):
    """Vanilla RNN for the Adding Problem."""

    def __init__(self, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(2, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.rnn(x)
        return torch.sigmoid(self.fc(h_n.squeeze(0)))


def train_adding(model, seq_len, n_train=3000, n_test=500, epochs=100, lr=0.001):
    """Train on Adding Problem and return MSE history."""
    X_tr, y_tr = generate_adding_problem(n_train, seq_len)
    X_te, y_te = generate_adding_problem(n_test, seq_len)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    mse_history = []

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            pred = model(X_te)
            mse = criterion(pred, y_te).item()
            mse_history.append(mse)

    return mse_history


# Compare RNN vs LSTM on Adding Problem at different sequence lengths
seq_lens_add = [50, 100, 200]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, sl in zip(axes, seq_lens_add):
    torch.manual_seed(42)

    rnn_mse = train_adding(AddingRNN(32), sl, epochs=80)
    lstm_mse = train_adding(AddingLSTM(32), sl, epochs=80)

    # Baseline: predicting 0.5 always (no memory)
    baseline_mse = 1.0 / 12.0  # Var of uniform on [0,1] ≈ 0.083

    ax.plot(rnn_mse, "r-", alpha=0.7, linewidth=2,
            label=f"RNN (final={rnn_mse[-1]:.4f})")
    ax.plot(lstm_mse, "g-", alpha=0.7, linewidth=2,
            label=f"LSTM (final={lstm_mse[-1]:.4f})")
    ax.axhline(baseline_mse, color="gray", linestyle=":", label="Baseline (predict 0.5)")
    ax.set_title(f"Seq length = {sl} (min lag = {sl//2})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(baseline_mse * 1.5, 0.1))

plt.suptitle("Adding Problem: RNN vs LSTM (Experiment 4)\n"
             "Adding Problem: RNN vs LSTM (논문 실험 4)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 6: Gradient Flow 실측 / Measuring Actual Gradient Flow

PyTorch autograd로 실제 기울기가 시간을 거슬러 전파될 때 RNN vs LSTM에서 어떻게 다른지 측정합니다.
Using PyTorch autograd to measure how gradients actually differ in RNN vs LSTM when propagating back through time.

In [ ]:
def measure_gradient_flow(model_class, seq_len, input_size=5, hidden_size=16):
    """Measure gradient magnitude at each time step for RNN/LSTM.

    Computes d(output_T) / d(input_t) for each t, showing how much
    the final output is sensitive to each input step.
    """
    torch.manual_seed(42)
    x = torch.randn(1, seq_len, input_size, requires_grad=True)

    if model_class == "rnn":
        rnn = nn.RNN(input_size, hidden_size, batch_first=True)
    else:
        rnn = nn.LSTM(input_size, hidden_size, batch_first=True)

    rnn.eval()
    output, _ = rnn(x)
    # Take the last hidden state's L2 norm as scalar output
    final_output = output[0, -1, :].norm()
    final_output.backward()

    # Gradient magnitude at each time step
    grad_norms = x.grad[0].norm(dim=1).detach().numpy()
    return grad_norms


seq_len = 100
rnn_grads = measure_gradient_flow("rnn", seq_len)
lstm_grads = measure_gradient_flow("lstm", seq_len)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Absolute gradient norms
axes[0].plot(range(seq_len), rnn_grads, "r-", alpha=0.7, linewidth=1.5, label="Vanilla RNN")
axes[0].plot(range(seq_len), lstm_grads, "g-", alpha=0.7, linewidth=1.5, label="LSTM")
axes[0].set_xlabel("Time step (t)")
axes[0].set_ylabel("$\\|\\frac{\\partial output_T}{\\partial input_t}\\|$")
axes[0].set_title("Gradient Magnitude at Each Time Step\n각 타임스텝에서의 기울기 크기")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Log scale
axes[1].semilogy(range(seq_len), rnn_grads + 1e-20, "r-", alpha=0.7, linewidth=1.5, label="Vanilla RNN")
axes[1].semilogy(range(seq_len), lstm_grads + 1e-20, "g-", alpha=0.7, linewidth=1.5, label="LSTM")
axes[1].set_xlabel("Time step (t)")
axes[1].set_ylabel("Gradient magnitude (log scale)")
axes[1].set_title("Same Data, Log Scale / 같은 데이터, 로그 스케일")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f"Actual Gradient Flow Measurement (seq_len={seq_len})\n"
             f"실제 기울기 흐름 측정 (시퀀스 길이={seq_len})", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key observation (핵심 관찰):")
print(f"  RNN:  gradient at t=0 / t={seq_len-1} = {rnn_grads[0] / (rnn_grads[-1] + 1e-20):.2e}")
print(f"  LSTM: gradient at t=0 / t={seq_len-1} = {lstm_grads[0] / (lstm_grads[-1] + 1e-20):.2e}")
print("  → RNN gradients decay drastically; LSTM maintains them!")

## Summary / 요약

### 이 구현에서 확인한 핵심 개념 / Key Concepts Verified

| 구현 / Implementation | 논문 섹션 / Section | 확인 내용 / What We Confirmed |
|---|---|---|
| Part 1: Vanishing gradient visualization | Section 3.1 (Eq. 2) | $f'_{\max} = 0.25$로 인한 기하급수적 감쇠, CEC는 일정 / Exponential decay from $f'_{\max}=0.25$, CEC is constant |
| Part 2: LSTM cell from scratch | Section 4 (Figure 1) | CEC 수식 $s(t) = s(t-1) + in \cdot g$ 직접 구현 / Direct implementation of CEC equation |
| Part 3: RNN vs LSTM comparison | Table 2 | LSTM은 긴 time lag에서 성공, RNN은 실패 / LSTM succeeds at long lags, RNN fails |
| Part 4: Gate dynamics | Section 4 | 게이트가 정보 흐름을 학습하여 제어 / Gates learn to control information flow |
| Part 5: Adding Problem | Experiment 4 (Table 7) | LSTM이 연속값 + 장기 저장 동시 해결 / LSTM handles continuous values + long-term storage |
| Part 6: Gradient flow measurement | Section 3.1 | 실측으로 RNN 기울기 소실 vs LSTM 유지 확인 / Measured RNN gradient vanishing vs LSTM preservation |

### 다음 논문과의 연결 / Connection to Next Papers

| 다음 논문 / Next Paper | LSTM과의 관계 / Relation to LSTM |
|---|---|
| **#10 LeCun et al. (1998) — LeNet-5** | CNN은 공간, LSTM은 시간. 나중에 CNN+LSTM 결합이 비디오/음성에서 강력해짐 / CNN for space, LSTM for time. CNN+LSTM later powerful for video/speech |
| **#17 Bahdanau et al. (2014) — Attention** | LSTM의 병목(고정 길이 hidden state)을 Attention이 보완. LSTM + Attention이 한동안 표준 / Attention complements LSTM's bottleneck. LSTM + Attention became standard |
| **#20 Vaswani et al. (2017) — Transformer** | 순환을 self-attention으로 완전 대체. LSTM의 순차 처리 한계를 병렬화로 해결 / Replaces recurrence with self-attention. Solves LSTM's sequential processing limit via parallelization |